## Importing Libraries

In [1]:
import numpy as np
import pandas as pd

## Importing Review Dataset

In [2]:
review_data = pd.read_csv('..\\Data\\Raw_Data\\googleplaystore_user_reviews.csv')
review_data.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462
2,10 Best Foods for You,NaN,NaN,NaN,NaN
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000


In [ ]:
# Creating a copy of the original dataframe to perform preprocessing steps
review_df = review_data.copy()
review_df.shape

(64295, 5)

In [ ]:
review_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 64295 entries, 0 to 64294
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   App                     64295 non-null  str    
 1   Translated_Review       37427 non-null  str    
 2   Sentiment               37432 non-null  str    
 3   Sentiment_Polarity      37432 non-null  float64
 4   Sentiment_Subjectivity  37432 non-null  float64
dtypes: float64(2), str(3)
memory usage: 8.3 MB


## PreProcessing Null Values

In [ ]:
# Checking for null values in the dataset
review_df.isna().sum()

App                           0
Translated_Review         26868
Sentiment                 26863
Sentiment_Polarity        26863
Sentiment_Subjectivity    26863
dtype: int64

In [7]:
# dropping rows with missing values in Translated review Column
review_df = review_df.dropna(subset=['Translated_Review'])

# filling missing values in Sentiment Column with Unknown
review_df["Sentiment"] = review_df["Sentiment"].fillna("Unknown")

# filling missing values in Sentiment_Polarity Column with median value
review_df["Sentiment_Polarity"] = review_df["Sentiment_Polarity"].fillna(review_df["Sentiment_Polarity"].median())

# filling missing values in Sentiment_Subjectivity Column with median value
review_df["Sentiment_Subjectivity"] = review_df["Sentiment_Subjectivity"].fillna(review_df["Sentiment_Subjectivity"].median())

In [ ]:
# validating the changes made to the dataset
print(review_df.isna().sum())
print(review_df.shape)

App                       0
Translated_Review         0
Sentiment                 0
Sentiment_Polarity        0
Sentiment_Subjectivity    0
dtype: int64
(37427, 5)


## PreProcessing Duplicated Values

In [ ]:
# checking for duplicate values in the dataset
print(review_df.duplicated().sum())

7735


In [ ]:
# dropping duplicate values from the dataset
review_df = review_df.drop_duplicates()

# validating the changes made to the dataset
print(review_df.duplicated().sum())

0


In [14]:
print(review_df.shape)

(29692, 5)


In [17]:
review_df.describe()

,Sentiment_Polarity,Sentiment_Subjectivity
count,29692.000000,29692.000000
mean,0.188868,0.490930
std,0.355694,0.265976
min,-1.000000,0.000000
25%,0.000000,0.350000
50%,0.157143,0.514286
75%,0.422917,0.652703
max,1.000000,1.000000


In [ ]:
# analyzing the distribution of the Sentiment column
review_df["Sentiment"].value_counts()

Sentiment
Positive    19015
Negative     6321
Neutral      4356
Name: count, dtype: int64

In [20]:
print(review_df.shape)
print(review_df.isna().sum())

(29692, 5)
App                       0
Translated_Review         0
Sentiment                 0
Sentiment_Polarity        0
Sentiment_Subjectivity    0
dtype: int64


## Feature Engineering

In [27]:
# Creating review length column
review_df["Review_length"] = review_df["Translated_Review"].apply(len)

# Creating word count column
review_df["Word_count"] = review_df["Translated_Review"].apply(lambda x: len(x.split()))

In [28]:
# Validating the changes made to the dataset
review_df[["Translated_Review", "Review_length", "Word_count"]].head()

,Translated_Review,Review_length,Word_count
0,I like eat delicious food. That's I'm cooking ...,122,21
1,This help eating healthy exercise regular basis,47,7
3,Works great especially going grocery store,42,6
4,Best idea us,12,3
5,Best way,8,2


In [29]:
# encode labels in Sentiment column
sentiment_map = {"Positive": 1, "Negative": 0, "Neutral": -1, "Unknown": 0}

review_df["Sentiment_Label"] = review_df["Sentiment"].map(sentiment_map)

In [30]:
# Creating Subjectivity Category column based on Sentiment_Subjectivity values
def subjectivity_category(value):
    if value < 0.3:
        return "Low"
    elif value < 0.7:
        return "Medium"
    else:
        return "High"
    
review_df["Subjectivity_Category"] = review_df["Sentiment_Subjectivity"].apply(subjectivity_category)

In [31]:
# Creating Polarity Category column based on Sentiment_Polarity values
def polarity_category(value):
    if value < 0:
        return "Positive"
    elif value < 0:
        return "Negative"
    else:
        return "Neutral"
    
review_df["Polarity_Type"] = review_df["Sentiment_Polarity"].apply(polarity_category)

In [32]:
# validating the changes made to the dataset
review_df.head()

,App,Translated_Review,Sentiment,Sentiment_Polarity,Sentiment_Subjectivity,Review_length,Word_count,Sentiment_Label,Subjectivity_Category,Polarity_Type
0,10 Best Foods for You,I like eat delicious food. That's I'm cooking ...,Positive,1.00,0.533333,122,21,1,Medium,Neutral
1,10 Best Foods for You,This help eating healthy exercise regular basis,Positive,0.25,0.288462,47,7,1,Low,Neutral
3,10 Best Foods for You,Works great especially going grocery store,Positive,0.40,0.875000,42,6,1,High,Neutral
4,10 Best Foods for You,Best idea us,Positive,1.00,0.300000,12,3,1,Medium,Neutral
5,10 Best Foods for You,Best way,Positive,1.00,0.300000,8,2,1,Medium,Neutral


In [34]:
review_df.describe()

,Sentiment_Polarity,Sentiment_Subjectivity,Review_length,Word_count,Sentiment_Label
count,29692.000000,29692.000000,29692.000000,29692.000000,29692.000000
mean,0.188868,0.490930,106.895224,17.332278,0.493702
std,0.355694,0.265976,102.600246,16.218162,0.737151
min,-1.000000,0.000000,2.000000,1.000000,-1.000000
25%,0.000000,0.350000,30.000000,5.000000,0.000000
50%,0.157143,0.514286,79.000000,13.000000,1.000000
75%,0.422917,0.652703,153.000000,25.000000,1.000000
max,1.000000,1.000000,2713.000000,345.000000,1.000000


In [35]:
# saving the cleaned dataset to a new CSV file
review_df.to_csv('..\Data\Cleaned_Data\cleaned_review_data.csv', index=False)

<>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<>:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
C:\Users\Om\AppData\Local\Temp\ipykernel_15936\4025591021.py:2: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  review_df.to_csv('..\Data\Cleaned_Data\cleaned_review_data.csv', index=False)
